# Grounding-line Position Recovery With Matched Filtering

This notebook revisits grounding-line motion using a more robust observable than independent arrival picks. Each synthetic is mapped into a **common field-coordinate frame**, and the reflected packet is measured by cross-correlating a time-windowed reflected packet against the baseline run.

This pass focuses on **larger offsets** (`0, 100, 250, 500 m`) to establish whether the reflected packet shift is monotonic before worrying about meter-scale sensitivity. The interpretation here is deliberately modest: the goal is to show **detectability**, **calibration**, and **residual bias**, not exact geometric recovery from a single uncalibrated formula.


In [ ]:
from csv import DictReader
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
summary_path = ROOT / 'results_glposition_large_10hz' / 'summary.csv'
requested_delta_x = np.asarray([0.0, 100.0, 250.0, 500.0])
rayleigh_speed_mps = 1900.0
x_targets_m = np.asarray([-1600.0, -1400.0, -1200.0, -1000.0, -800.0])
reflected_window_s = (6.3, 8.2)
max_lag_s = 0.8

def load_rows(path):
    with path.open() as handle:
        return list(DictReader(handle))

rows = sorted(load_rows(summary_path), key=lambda row: float(row['delta_x_m']))
completed_delta_x = np.asarray([float(row['delta_x_m']) for row in rows])
pending_delta_x = [dx for dx in requested_delta_x if dx not in set(completed_delta_x)]
print(f'Loaded {len(rows)} completed large-offset cases')
print('Completed offsets:', completed_delta_x.tolist())
print('Pending offsets:', pending_delta_x if pending_delta_x else 'none')
rows

In [ ]:
def load_gather(case_id):
    data = np.load(ROOT / 'results_glposition_large_10hz' / case_id / 'surface_gather.npz', allow_pickle=True)
    return {key: data[key] for key in data.files}

def field_x(gather, row):
    return gather['x'] - float(row['coordinate_shift_m'])

def trace_at_field_x(gather, row, x_target_m):
    x_field = field_x(gather, row)
    idx = int(np.argmin(np.abs(x_field - x_target_m)))
    return idx, x_field[idx], gather['bxx'][idx, :]

def analytic_envelope(data):
    data = np.asarray(data, dtype=float)
    n = data.size
    spectrum = np.fft.fft(data)
    h = np.zeros(n)
    if n % 2 == 0:
        h[0] = 1.0
        h[n // 2] = 1.0
        h[1:n // 2] = 2.0
    else:
        h[0] = 1.0
        h[1:(n + 1) // 2] = 2.0
    return np.abs(np.fft.ifft(spectrum * h))

def cosine_taper(n, fraction=0.15):
    n_taper = max(1, min(int(round(fraction * n)), n // 2))
    window = np.ones(n)
    ramp = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, n_taper)))
    window[:n_taper] = ramp
    window[-n_taper:] = ramp[::-1]
    return window

def reflected_packet(trace, time, window_s):
    mask = (time >= window_s[0]) & (time <= window_s[1])
    packet = np.asarray(trace[mask], dtype=float).copy()
    packet -= packet.mean()
    packet *= cosine_taper(len(packet), fraction=0.12)
    return time[mask], packet

def best_lag_seconds(reference, candidate, dt_s, max_lag_s):
    ref = analytic_envelope(reference)
    cand = analytic_envelope(candidate)
    max_lag = int(round(max_lag_s / dt_s))
    best = None
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            a = ref[: len(ref) - lag]
            b = cand[lag:]
        else:
            a = ref[-lag:]
            b = cand[: len(cand) + lag]
        if len(a) < 10:
            continue
        corr = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
        if best is None or corr > best[0]:
            best = (corr, lag)
    return best[1] * dt_s, best[0]


In [ ]:
baseline_row = rows[0]
baseline_gather = load_gather(baseline_row['case_id'])
baseline_time = baseline_gather['time']
dt_s = float(np.median(np.diff(baseline_time)))

records = []
for row in rows:
    gather = load_gather(row['case_id'])
    lags = []
    corrs = []
    for x_target in x_targets_m:
        _, _, base_trace = trace_at_field_x(baseline_gather, baseline_row, x_target)
        _, _, test_trace = trace_at_field_x(gather, row, x_target)
        _, base_packet = reflected_packet(base_trace, baseline_time, reflected_window_s)
        _, test_packet = reflected_packet(test_trace, gather['time'], reflected_window_s)
        lag_s, corr = best_lag_seconds(base_packet, test_packet, dt_s=dt_s, max_lag_s=max_lag_s)
        lags.append(lag_s)
        corrs.append(corr)
    lags = np.asarray(lags)
    corrs = np.asarray(corrs)
    mean_lag_s = np.average(lags, weights=np.clip(corrs, 1e-6, None))
    recovered_delta_x_m = -0.5 * rayleigh_speed_mps * mean_lag_s
    records.append({
        'case_id': row['case_id'],
        'delta_x_m': float(row['delta_x_m']),
        'mean_lag_s': mean_lag_s,
        'recovered_delta_x_m': recovered_delta_x_m,
        'mean_corr': float(np.mean(corrs)),
        'lags_s': lags,
        'corrs': corrs,
    })

for rec in records:
    print(
        f"{rec['case_id']:>8s}  true Δx={rec['delta_x_m']:7.1f} m  "
        f"raw lag={rec['mean_lag_s']: .4f} s  recovered Δx={rec['recovered_delta_x_m']:7.1f} m  "
        f"mean corr={rec['mean_corr']:.3f}"
    )


In [ ]:
delta_x = np.asarray([rec['delta_x_m'] for rec in records])
recovered_delta_x = np.asarray([rec['recovered_delta_x_m'] for rec in records])
raw_mean_lag = np.asarray([rec['mean_lag_s'] for rec in records])
positive_delay = -raw_mean_lag
mean_corr = np.asarray([rec['mean_corr'] for rec in records])
dx_error = recovered_delta_x - delta_x
percent_error = np.full_like(delta_x, np.nan, dtype=float)
mask_nonzero = delta_x > 0.0
percent_error[mask_nonzero] = 100.0 * dx_error[mask_nonzero] / delta_x[mask_nonzero]
rms_error = np.sqrt(np.mean(dx_error[mask_nonzero]**2))
fit_slope, fit_intercept = np.polyfit(delta_x, recovered_delta_x, 1)
calibrated_delta_x = (recovered_delta_x - fit_intercept) / fit_slope
calibrated_error = calibrated_delta_x - delta_x
calibrated_rms_error = np.sqrt(np.mean(calibrated_error[mask_nonzero]**2))

print(f'Uncalibrated RMS recovery error = {rms_error:.2f} m')
print(f'Best-fit calibration: recovered ≈ {fit_slope:.3f} * true + {fit_intercept:.2f} m')
print(f'Calibrated RMS error over nonzero offsets = {calibrated_rms_error:.2f} m')
print()
for dx_true, dx_rec, err, pct, dx_cal, err_cal in zip(
    delta_x, recovered_delta_x, dx_error, percent_error, calibrated_delta_x, calibrated_error
):
    pct_text = 'n/a' if not np.isfinite(pct) else f'{pct: .1f}%'
    print(
        f'true Δx = {dx_true:7.1f} m   recovered = {dx_rec:7.1f} m   '
        f'error = {err: .1f} m   percent error = {pct_text:>6s}   '
        f'calibrated Δx = {dx_cal:7.1f} m   calibrated error = {err_cal: .1f} m'
    )


## Interpretation

The key result is that the reflected-packet delay is **monotonic** with imposed grounding-line offset once the comparison is made in a common field frame and the reflected packet is measured by matched filtering. The remaining mismatch should be interpreted as **bias in a band-limited packet observable**, not as simple FFT resolution error. In practice this means the method is most useful as a **calibrated tracking observable** rather than an exact geometric inversion formula.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(delta_x, positive_delay, 'o-', lw=2, ms=8, label='matched-filter delay')
ax.plot(delta_x, 2.0 * delta_x / rayleigh_speed_mps, 'k--', lw=1.5, label='2Δx / c_R')
ax.set_xlabel('true grounding-line offset Δx (m)')
ax.set_ylabel('positive delay of reflected packet (s)')
ax.set_title('Reflected-packet delay in the common field frame')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(delta_x, recovered_delta_x, 'o-', lw=2, ms=8, label='matched-filter recovery')
fit_x = np.linspace(0.0, requested_delta_x.max(), 200)
ax.plot(fit_x, fit_slope * fit_x + fit_intercept, color='tab:orange', lw=2, label='best-fit calibration')
ax.plot(fit_x, fit_x, 'k--', lw=1.5, label='1:1')
ax.set_xlabel('true grounding-line offset Δx (m)')
ax.set_ylabel('recovered Δx from matched filter (m)')
ax.set_title('Grounding-line offset recovery before calibration')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].axhline(0.0, color='k', lw=1.0, alpha=0.5)
axes[0].plot(delta_x, dx_error, 'o-', lw=2, ms=8, label='uncalibrated residual')
axes[0].plot(delta_x, calibrated_error, 's--', lw=1.5, ms=7, label='calibrated residual')
axes[0].set_xlabel('true grounding-line offset Δx (m)')
axes[0].set_ylabel('recovery error (m)')
axes[0].set_title('Residual bias')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(delta_x, calibrated_delta_x, 'o-', lw=2, ms=8, label='after calibration')
axes[1].plot(fit_x, fit_x, 'k--', lw=1.5, label='1:1')
axes[1].set_xlabel('true grounding-line offset Δx (m)')
axes[1].set_ylabel('calibrated recovered Δx (m)')
axes[1].set_title('Empirical calibration view')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle('Use this as a calibrated tracking observable, not an exact geometric inversion')


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(delta_x, mean_corr, 's-', color='tab:red', lw=1.5, label='mean matched-filter corr')
ax1.set_xlabel('true grounding-line offset Δx (m)')
ax1.set_ylabel('mean envelope correlation', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
summary_rows = {float(row['delta_x_m']): row for row in rows}
reflection = np.asarray([float(summary_rows[dx]['reflection_coefficient']) for dx in delta_x])
ax2.plot(delta_x, reflection, 'o--', color='tab:blue', lw=1.5, label='reflection coefficient')
ax2.set_ylabel('reflection coefficient', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')
fig.suptitle('Matched-filter quality and reflection-amplitude context')
fig.tight_layout()


In [ ]:
selected_x = -1000.0
fig, axes = plt.subplots(len(rows), 1, figsize=(10, 2.7 * len(rows)), sharex=True, constrained_layout=True)
for ax, row in zip(np.atleast_1d(axes), rows):
    gather = load_gather(row['case_id'])
    x_field = field_x(gather, row)
    idx = int(np.argmin(np.abs(x_field - selected_x)))
    trace = gather['bxx'][idx] / max(np.max(np.abs(gather['bxx'][idx])), 1e-12)
    ax.plot(gather['time'], trace, color='black', lw=1.0)
    ax.axvspan(*reflected_window_s, color='tab:orange', alpha=0.15)
    ax.set_title(f"Δx = {float(row['delta_x_m']):.1f} m, trace near field x = {x_field[idx]:.1f} m")
    ax.grid(True, alpha=0.25)
    ax.set_ylabel('norm. BXX')
axes[-1].set_xlabel('time (s)')


In [ ]:
fig, axes = plt.subplots(1, len(rows), figsize=(5 * max(1, len(rows)), 5), sharey=True, constrained_layout=True)
for ax, row in zip(np.atleast_1d(axes), rows):
    gather = load_gather(row['case_id'])
    x_field = field_x(gather, row)
    clip = np.percentile(np.abs(gather['bxx']), 99)
    ax.imshow(
        gather['bxx'].T,
        origin='lower',
        aspect='auto',
        extent=[x_field.min(), x_field.max(), gather['time'].min(), gather['time'].max()],
        vmin=-clip, vmax=clip,
    )
    ax.axvspan(*reflected_window_s, color='white', alpha=0.05)
    ax.set_title(f"Δx = {float(row['delta_x_m']):.1f} m")
    ax.set_xlabel('field x (m)')
axes[0].set_ylabel('time (s)')
fig.suptitle('Surface gathers in the common field-coordinate frame')